In [1]:
import torch 
print(torch.cuda.is_available())
print(torch.__version__)

True
2.6.0+cu124


In [2]:
!nvidia-smi

Mon Mar  2 14:12:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.163.01             Driver Version: 550.163.01     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA TITAN Xp                Off |   00000000:02:00.0  On |                  N/A |
| 28%   40C    P8             17W /  250W |     282MiB /  12288MiB |     22%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Load the validation and test dataset

In [3]:
import pandas as pd 
import numpy as np 
from matplotlib import pyplot as plt
from collections import Counter

df = pd.read_csv('../data/dontpatronizeme_pcl.tsv', sep='\t')

# Remove rows with NA labels 
df = df.dropna() 

# Add a bool_labels column for binary classification
df["bool_labels"] = df["label"] > 1   # is PCL if >1
val_labels = pd.read_csv('../data/dev_semeval_parids-labels.csv')["par_id"]  # extract the labels for val dataset 
df_val = df[df["par_id"].isin(val_labels)].reset_index() 

# read the test dataset 
df_test = pd.read_csv('../data/task4_test.tsv', sep='\t')


In [23]:
print(df_val.shape[0]) 
print(df_test.shape[0])

2093
3832


# Text Cleaning

In [4]:
# Remove special characters
SPECIAL_CHARACTERS = ['&amp;', '&lt;', '&gt;', '<h>', '\n', '\t']
for char in SPECIAL_CHARACTERS:
    df_val["text"] = df_val["text"].str.replace(char, "")
    df_test["text"] = df_test["text"].str.replace(char, "")

# Replace numbers with 0
df_val["text"] = df_val["text"].str.replace(r"\d+", "0", regex=True)
df_test["text"] = df_test["text"].str.replace(r"\d+", "0", regex=True)


# Add contextual information to the text tokens

In [5]:
def add_info(df): 
    # Append the keyword and country code to the text, and separate them with additional separator tokens
    # Remove dashes in the keyword to match the format in the texts 
    return df["keyword"].str.replace('-', " ") + "<SEP>" + df["country_code"] + "<SEP>" + df["text"]

# Tokenization

In [6]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification, BertTokenizer, BertForSequenceClassification, AutoConfig, Trainer, TrainingArguments

tokenizer_roberta = RobertaTokenizer.from_pretrained("roberta-base") 
tokenizer_bert = BertTokenizer.from_pretrained("bert-base-uncased") 

# define special tokens for separating text 
special_tokens = {"additional_special_tokens": ["<SEP>"]}
tokenizer_roberta.add_special_tokens(special_tokens) 
tokenizer_bert.add_special_tokens(special_tokens) 

# Create text with contextual information 
def tokenize(tokenizer, df): 
    text_with_context = add_info(df) 

    encoding = tokenizer(
        text_with_context.tolist(), 
        padding="max_length",   # Add padding to shorter sentences 
        max_length=256,
        truncation = True, 
        return_attention_mask = True 
    )

    return encoding

/vol/bitbucket/bm1325/dl_cw_1/dlvenv/lib/python3.12/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/vol/bitbucket/bm1325/dl_cw_1/dlvenv/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/vol/bitbucket/bm1325/dl_cw_1/dlvenv/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


# Convert to pyTorch dataset

In [7]:
import torch 
from torch.utils.data import DataLoader, TensorDataset
from datasets import Dataset

def to_dataset_val(tokenizer, df): 
    # Obtain tokens (input_ids, attention_mask) from the dataset 
    encoding = tokenize(tokenizer, df) 

    # Return huggingface dataset 
    return Dataset.from_dict({
        "input_ids": encoding["input_ids"], 
        "attention_mask": encoding["attention_mask"], 
        "label": df["bool_labels"].values 
    })

# the test set does not have labels 
def to_dataset_test(tokenizer, df): 
    # Obtain tokens (input_ids, attention_mask) from the dataset 
    encoding = tokenize(tokenizer, df) 

    # Return huggingface dataset 
    return Dataset.from_dict({
        "input_ids": encoding["input_ids"], 
        "attention_mask": encoding["attention_mask"]
    })

In [8]:
val_dataset_r = to_dataset_val(tokenizer_roberta, df_val) 
val_dataset_b = to_dataset_val(tokenizer_bert, df_val) 
test_dataset_r = to_dataset_test(tokenizer_roberta, df_test) 
test_dataset_b = to_dataset_test(tokenizer_bert, df_test) 

# set to torch format 
val_dataset_r.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_dataset_b.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset_r.set_format("torch", columns=["input_ids", "attention_mask"])
test_dataset_b.set_format("torch", columns=["input_ids", "attention_mask"])

# Load both BERT and RoBERTa models
They were trained on data formatted the same way: 
- Text cleaning
- Oversample the minority class for each keyword
- Add contextual information including the keyword and the country code to the text 

In [9]:
import os 

# must pass abs path here 
bert_model = BertForSequenceClassification.from_pretrained(os.path.abspath("/vol/bitbucket/bm1325/NLP-PCL-Classification/models/model_bert_ensemble"), local_files_only = True)
roberta_model = RobertaForSequenceClassification.from_pretrained(os.path.abspath("/vol/bitbucket/bm1325/NLP-PCL-Classification/models/model_roberta_ensemble"), local_files_only = True)

trainer_bert = Trainer(model = bert_model)
trainer_roberta = Trainer(model = roberta_model)

# Grid search on the validation dataset

In [10]:
# convert loaded models to Trainer objects 
trainer_r = Trainer(model = roberta_model)
trainer_b = Trainer(model = bert_model)

# Make predictions on the validation dataset using the roberta model (r) and bert model (b) respectively 
val_pred_r = trainer_r.predict(val_dataset_r) 
val_pred_b = trainer_b.predict(val_dataset_b)

  0%|          | 0/262 [00:00<?, ?it/s]

  0%|          | 0/262 [00:00<?, ?it/s]

In [11]:
from sklearn.metrics import f1_score

y_true = df_val["bool_labels"]

# x refers to the weight of the roberta model 
# 0 = pure BERT, 1 = pure RoBERTa (Improved version) 
x_values = np.arange(0, 1.05, 0.05)
best_x = -1 
cur_f1 = 0 

for x in x_values: 
    y_pred = val_pred_r.predictions*x + val_pred_b.predictions*(1-x) 

    # Extract the most likely output 
    y_pred = y_pred.argmax(axis=1) 

    # Calculate f1 score 
    f1 = f1_score(y_pred, y_true)
    if f1 > cur_f1: 
        cur_f1 = f1 
        best_x = x 
    print(f1) 

print("="*50)
print(f"Best x: {best_x}")
print(f"Best f1: {cur_f1}")

0.5511811023622047
0.5549738219895288
0.5535248041775457
0.5558441558441558
0.561038961038961
0.5625
0.5647668393782384
0.5670103092783505
0.5846153846153846
0.6030150753768844
0.6105769230769231
0.6291079812206573
0.628175519630485
0.6292134831460674
0.6261261261261262
0.6205357142857143
0.6299559471365639
0.6244541484716157
0.6220302375809935
0.6193548387096774
0.6193548387096774
Best x: 0.8
Best f1: 0.6299559471365639


# Use the best x value to make predictions on the val and test set

In [24]:
val_pred = val_pred_r.predictions*best_x + val_pred_b.predictions*(1-best_x) 
val_pred = val_pred.argmax(axis=1) 

test_pred_r = trainer_r.predict(test_dataset_r) 
test_pred_b = trainer_b.predict(test_dataset_b)
test_pred = test_pred_r.predictions*best_x + test_pred_b.predictions*(1-best_x) 
test_pred = test_pred.argmax(axis=1) 

print(val_pred) 
print(test_pred) 

  0%|          | 0/479 [00:00<?, ?it/s]

  0%|          | 0/479 [00:00<?, ?it/s]

[0 0 0 ... 0 0 1]
[0 0 0 ... 0 0 0]


In [32]:
# write to dev.txt and test.txt 
with open("dev.txt", "w") as f: 
    f.write('\n'.join(str(x) for x in val_pred)) 

with open("test.txt", "w") as f: 
    f.write('\n'.join(str(x) for x in test_pred)) 


In [40]:
print(len(val_pred)) 
print(len(test_pred))

# Verify that the val f1 score is 0.629
print(f1_score(df_val["bool_labels"], val_pred))

2093
3832
0.6299559471365639
